# Semantic Search & LLM Application

FAQ e-commerce semantic search dengan Gemini Embeddings, FAISS, dan jawaban Gemini berbasis context.

## 1. Load knowledge base

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import semantic_search as app

DATA_PATH = PROJECT_DIR / "data" / "ecommerce_faq_knowledge_base.csv"
INDEX_PATH = PROJECT_DIR / "artifacts" / "ecommerce_faq.faiss"
df = app.load_knowledge_base(DATA_PATH)
print("Total entri:", len(df))
print("Kolom:", list(df.columns))
display(df["category"].value_counts().rename_axis("kategori").to_frame("jumlah"))

Total entri: 59
Kolom: ['faq_id', 'category', 'question', 'answer']


,jumlah
kategori,
Pemesanan,9
Pengiriman,9
Pembayaran,8
Pengembalian,8
Akun,7
Produk,7
Promo,6
Customer Service,5


## 2. Embed dan simpan ke FAISS

In [2]:
documents = app.make_retrievable_texts(df)
client = app.get_client()

print(f"Membuat {len(documents)} embedding dengan {app.EMBEDDING_MODEL}. Proses memakai Gemini API dan dapat memerlukan sekitar satu menit.")
embeddings = app.embed_texts(documents, client=client)
assert embeddings.shape == (len(df), app.EMBEDDING_DIMENSION)
print("Shape embeddings:", embeddings.shape)

index = app.build_faiss_index(embeddings)
app.save_faiss_index(index, df, INDEX_PATH)
index = app.load_faiss_index(INDEX_PATH, df)
assert index.ntotal == len(df)
print("Vektor FAISS:", index.ntotal)

Membuat 59 embedding dengan gemini-embedding-2. Proses memakai Gemini API dan dapat memerlukan sekitar satu menit.
Shape embeddings: (59, 768)
Vektor FAISS: 59


## 3. Semantic search, prompt, dan jawaban LLM

In [3]:
def semantic_search(query, top_k=3):
    return app.semantic_search(query, df, index, client=client, top_k=top_k)

def generate_answer(query, top_k=3):
    return app.answer_question(query, df, index, client=client, top_k=top_k)

test_results = semantic_search("kenapa paket saya belum sampai?")
for result in test_results:
    print(f"FAQ {result['faq_id']} | cosine={result['similarity']:.4f} | {result['question']}")

print()
print(app.build_prompt("kenapa paket saya belum sampai?", test_results))

example = generate_answer("apakah bisa bayar pakai cicilan?")
print("Jawaban LLM:", example["answer"])

FAQ 12 | cosine=0.7793 | Bagaimana jika paket saya belum sampai melebihi estimasi?
FAQ 14 | cosine=0.7437 | Kenapa status pengiriman saya tidak berubah selama beberapa hari?
FAQ 52 | cosine=0.7304 | Kenapa nomor resi saya belum bisa dilacak di website kurir?

Persona:
Kamu adalah asisten customer service e-commerce. Jawabanmu ramah, singkat, dan hanya berdasarkan FAQ yang diberikan.

Task:
Jawab pertanyaan pengguna hanya dari Context. Jangan menambah kebijakan, syarat, atau janji yang tidak tertulis di Context. Jika Context tidak menjawab pertanyaan, katakan bahwa informasi tersebut belum tersedia di knowledge base dan sarankan pengguna menghubungi customer service.

Context:
FAQ 12 (Pengiriman)
Pertanyaan: Bagaimana jika paket saya belum sampai melebihi estimasi?
Jawaban: Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.

FAQ 14 (Pengiriman)
Pertanyaan: Kenapa status pengiriman saya 

## 4. Uji relevansi

In [4]:
test_queries = [
    ("Langsung", "Bagaimana cara melacak paket saya?"),
    ("Langsung", "Apakah bisa bayar pakai cicilan?"),
    ("Parafrasa", "Barang saya sudah melewati perkiraan tiba, harus bagaimana?"),
    ("Parafrasa", "Saya menerima barang yang salah, bagaimana cara mengajukan pengembalian?"),
    ("Di luar knowledge base", "Bisakah teknisi memasang mesin cuci yang saya beli di rumah?"),
]

print("MIN_SIMILARITY:", app.get_min_similarity())
for kind, query in test_queries:
    result = generate_answer(query)
    print(f"\n{kind}: {query}")
    for item in result["search_results"]:
        used = "context" if item in result["context_results"] else "di bawah ambang"
        print(f"  FAQ {item['faq_id']} | {item['similarity']:.4f} | {used}")
    print("Jawaban:", result["answer"])

MIN_SIMILARITY: 0.65

Langsung: Bagaimana cara melacak paket saya?
  FAQ 10 | 0.7908 | context
  FAQ 12 | 0.7259 | context
  FAQ 6 | 0.7259 | context
Jawaban: Halo! Nomor resi akan muncul di halaman 'Pesanan Saya' setelah barang dikirim, dan Anda dapat menggunakannya untuk melacak posisi paket di website kurir terkait.

Langsung: Apakah bisa bayar pakai cicilan?
  FAQ 20 | 0.8342 | context
  FAQ 22 | 0.7370 | context
  FAQ 17 | 0.7323 | context
Jawaban: Halo! Ya, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank. Jika ada hal lain yang ingin ditanyakan, silakan beri tahu ya!

Parafrasa: Barang saya sudah melewati perkiraan tiba, harus bagaimana?
  FAQ 12 | 0.8149 | context
  FAQ 14 | 0.7122 | context
  FAQ 9 | 0.7000 | context
Jawaban: Halo! Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service kami dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.

Parafrasa: Saya menerim